#### Data Generating Process with known causal effect

In [24]:
import numpy as np
import pandas as pd

def generate_synthetic_data(n_samples, gamma, effect_type="linear", seed=42):
    np.random.seed(seed)
    x1 = np.random.normal(loc=0, scale=1, size=n_samples)
    x2 = np.random.normal(loc=0, scale=1, size=n_samples)
    
    logit = -gamma * x1
    e_x = 1 / (1 + np.exp(-logit))
    w = np.random.binomial(1, e_x, size=n_samples)
    
    eps = np.random.normal(loc=0, scale=1.0, size=n_samples)
    eta = np.random.normal(loc=0, scale=0.5, size=n_samples)
    
    y0 = 2 * x1 + np.sin(x2) + eps
    
    if effect_type == "complex":
        # Highly non-linear HTE that penalizes S-Learner regularization shrinkage
        tau = 2.0 + (x1 ** 2) * np.sin(x2) * 2.0
    else:
        # Standard linear HTE
        tau = 1.5 + 2 * x2
        
    y = y0 + w * tau + eta
    
    df = pd.DataFrame({
        'x1': x1, 'x2': x2, 'e_x': e_x,
        'w': w, 'y0': y0, 'tau': tau, 'y': y
    })
    return df

#### Modelling the S-T-X Learners

In [6]:
# S-Learner 

from sklearn.ensemble import RandomForestRegressor
import numpy as np


def train_s_learner(df):
    # prepare features
    X_with_w = df[['x1','x2','w']].values
    y = df['y'].values

    rf = RandomForestRegressor(max_depth=10, random_state=0)
    rf.fit(X_with_w,y)

    X_w1 = X_with_w.copy()
    X_w1[:,2] = 1.0 

    X_w0 = X_with_w.copy()
    X_w0[:,2] = 0.0 

    pred_w1 = rf.predict(X_w1)
    pred_w0 = rf.predict(X_w0)

    tau_hat = pred_w1 - pred_w0

    return tau_hat

In [7]:
tau_predictions = train_s_learner(df)

In [ ]:
def train_t_learner(df):
     w = df['w'].values
    y = df  X = df[['x1','x2']].values
    y = ['y'].values

    X_treated = X[w == 1]
    y_treated = y[w == 1]

    X_control = X[w==0]
    y_control = y[w==0]

    model_1 = RandomForestRegressor(max_depth=10, random_state=0)
    model_0 = RandomForestRegressor(max_depth=10, random_state=0)

    model_1.fit(X_treated, y_treated)
    model_0.fit(X_control,y_control)

    pred_1 = model_1.predict(X)
    pred_0 = model_0.predict(X)

    tau_hat = pred_1 - pred_0

    return tau_hat

In [13]:
tau_predictions_t = train_t_learner(df)

In [16]:
from sklearn.linear_model import LogisticRegression

def train_x_learner(df):
    w = df['w'].values
    X = df[['x1','x2']].values
    y = df['y'].values

    X_treated = X[w == 1]
    y_treated = y[w == 1]

    X_control = X[w==0]
    y_control = y[w==0]

    model_1 = RandomForestRegressor(max_depth=10, random_state=0)
    model_0 = RandomForestRegressor(max_depth=10, random_state=0)

    model_1.fit(X_treated, y_treated)
    model_0.fit(X_control,y_control)

    d_1 = y_treated - model_0.predict(X_treated)
    d_0 = model_1.predict(X_control) - y_control

    tau_model_1 = RandomForestRegressor(max_depth=10, random_state=0)
    tau_model_0 = RandomForestRegressor(max_depth=10, random_state=0)
    tau_model_0.fit(X_control, d_0)
    tau_model_1.fit(X_treated, d_1)

    ## Train the propensity score model e(X) using logistic regression

    ps_model = LogisticRegression()
    ps_model.fit(X,w)
    e_x = ps_model.predict_proba(X)[:,1]

    tau_hat = e_x * tau_model_0.predict(X) + (1-e_x) * tau_model_1.predict(X)

    return tau_hat

In [17]:
tau_predictions_x = train_x_learner(df)

#### Define Precision in Estimation of Heterogeneous Effects validation

In [18]:
def calculate_pehe(tau_true, tau_pred):
    return np.sqrt(np.mean((tau_true - tau_pred) ** 2))

#### Compare across various scenarios

In [26]:
scenarios = [
    {"n": 200,  "gamma": 0.5, "effect": "linear",  "label": "Small N, Balanced"},
    {"n": 2000, "gamma": 0.5, "effect": "linear",  "label": "Large N, Balanced"},
    {"n": 5000, "gamma": 0.5, "effect": "complex", "label": "Large N, Complex HTE (T-Learner Wins)"},
    {"n": 2000, "gamma": 3.5, "effect": "linear",  "label": "Large N, High Positivity Strain"},
]

results = []

for sc in scenarios:
    # 1. Generate data passing the effect type
    df_sim = generate_synthetic_data(
        n_samples=sc["n"], 
        gamma=sc["gamma"], 
        effect_type=sc["effect"], 
        seed=42
    )
    tau_true = df_sim['tau'].values

    # 2. Estimate effects
    tau_s = train_s_learner(df_sim)
    tau_t = train_t_learner(df_sim)
    tau_x = train_x_learner(df_sim)

    # 3. Calculate PEHE
    results.append({
        "Scenario": sc["label"],
        "S-Learner PEHE": round(calculate_pehe(tau_true, tau_s), 4),
        "T-Learner PEHE": round(calculate_pehe(tau_true, tau_t), 4),
        "X-Learner PEHE": round(calculate_pehe(tau_true, tau_x), 4)
    })

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(pd.DataFrame(results))

                                Scenario  S-Learner PEHE  T-Learner PEHE  X-Learner PEHE
0                      Small N, Balanced          1.3354          1.1508          0.8104
1                      Large N, Balanced          0.7235          0.8478          0.5769
2  Large N, Complex HTE (T-Learner Wins)          0.7274          0.7080          0.5585
3        Large N, High Positivity Strain          0.9610          1.2472          0.5806
